# Proyecto Dijkstra y Circuitos Hamiltonianos — Farmacias Simán SPS

Este notebook está organizado por etapas para que el código sea más fácil de ejecutar y explicar.

**Estructura:**
1. Configuración y datos
2. Construcción del grafo
3. Dijkstra
4. Dirac y Ore
5. Backtracking Hamiltoniano
6. Resultados Hamiltonianos
7. Análisis de los candidatos con Dijkstra
8. Resumen final


## 1. Importaciones y funciones auxiliares

In [1]:
from pathlib import Path
from math import inf
import heapq
import openpyxl

ARCHIVO_EXCEL=Path("Datos_SucursalesSimán.xlsx")

#convierte el valor del excel en float
def numero(valor):
    if valor is None:
        return None
    
    try:
        
        return float(valor)
    
    except(TypeError,ValueError):
        return None
    
    
def formato_km(valor):
    if valor is None or valor==inf:
        return "N/D"
    
    return f"{valor:.2f} km"

## 2. Datos de los candidatos

In [2]:
CANDIDATOS={
    
    "A": {
        "nombre": "Candidato A",
        "zona": "Zona Norte",
        "ubicacion": "Sector La 105 Brigada",
    },
    "B": {
        "nombre": "Candidato B",
        "zona": "Zona Sur",
        "ubicacion": "Zona industrial, Sector el Polvorín",
    },
    "C": {
        "nombre": "Candidato C",
        "zona": "Zona Centro",
        "ubicacion": "Barrio Los Andes, 21102 San Pedro Sula, Cortés, Honduras",
    },
    
}


def nombre_candidato(letra):
    
    return CANDIDATOS[str(letra).upper()]["nombre"]

def datos_candidato(letra):
    return CANDIDATOS[str(letra).upper()]

if not ARCHIVO_EXCEL.exists():
    
    raise FileNotFoundError(f"No se encontro el archivo: {ARCHIVO_EXCEL.resolve()}")

wb=openpyxl.load_workbook(ARCHIVO_EXCEL,data_only=True)

## 3. Carga y validación de los 5 bloques

In [3]:
#leer los 5 bloques, Lee la hoja 'Bloques de Sucursales' dentro del excel
def cargar_bloques(hoja):
    
    bloques={"Bloque 1":[],"Bloque 2":[],"Bloque 3":[],"Bloque 4":[],"Bloque 5":[],}
    
    #bloques 1,2,3
    for fila in range(3,15):
        columnas=[(2,"Bloque 1"),(5,"Bloque 2"),(8,"Bloque 3"),]
        
        for columna_nombre,bloque in columnas:
            nombre=hoja.cell(fila,columna_nombre).value
            if nombre:
                bloques[bloque].append(str(nombre).strip())
                
    #bloque 4,5
    for fila in range(19,31):
        columnas=[(2,"Bloque 4"),(5,"Bloque 5")]
        
        for columna_nombre,bloque in columnas:
                    nombre=hoja.cell(fila,columna_nombre).value
                    if nombre:
                        bloques[bloque].append(str(nombre).strip())
                        
    return bloques
        
        
hoja_bloques=wb["Bloques de Sucursales"]
bloques=cargar_bloques(hoja_bloques)

#verificar q cada bloque debe tener 12 farmacias 
for bloque,farmacias in bloques.items():
    
    if len(farmacias)!=12:
        raise ValueError(f"{bloque} tiene {len(farmacias)} farmacias; se esperan 12")
    

## 4. Construcción del grafo

In [4]:
#Aca se construyen el grafo de las distancias vecinas 
#Se guarda como conexion no dirigida porque el peso de Dijkstra sera el promedio ida/vuelta. Esto permite que todas las farmacias del bloque sean alcanzables.    

grafo={}

def agregar_vertice(nombre):
    if nombre not in grafo:
        grafo[nombre]={}
    
#Agrega una conexion con pesos ida, vuelta y promedio    
def agregar_conexion(a,b,ida,vuelta):
    
    ida=numero(ida)
    vuelta=numero(vuelta)
    
    if ida is None or vuelta is None:
        return
    
    promedio=(ida+vuelta)/2
    agregar_vertice(a)
    agregar_vertice(b)
    
    datos={"ida":ida,"vuelta":vuelta,"peso":promedio}
    
    #si la misma conexion apareciera mas de una vez, conservarla de menor promedio.
    anterior=grafo[a].get(b)
    if anterior is None or promedio<anterior["peso"]:
        
        grafo[a][b]=datos
        #A -> B
        #ida = 2.0
        #vuelta = 3.0

        #B -> A
        #ida = 3.0
        #vuelta = 2.0
        grafo[b][a]={"ida":vuelta,"vuelta":ida,"peso":promedio}
        
        

## 5. Conexiones de los candidatos

In [5]:
hojas_vecinos=wb["Distancias de Farmacias Vecinas"]
farmacia_actual=None

for fila in range(2,hojas_vecinos.max_row+1):
    numero_farmacia=hojas_vecinos.cell(fila,1).value
    nombre=hojas_vecinos.cell(fila,2).value
    
    #una fila con numero de farmacia marca el inicio de un grupo
    if numero_farmacia is not None and nombre:
        farmacia_actual=str(nombre).strip()
        agregar_vertice(farmacia_actual)
        continue
    
    #las siguientes filas contienen sus 3 farmacias vecinas
    if farmacia_actual and nombre:
        vecino=str(nombre).strip()
        ida=hojas_vecinos.cell(fila,4).value
        vuelta=hojas_vecinos.cell(fila,5).value
        
        agregar_conexion(farmacia_actual,vecino,ida,vuelta)
        
        
#leer y agregar los 3 candidatos 

Candidato_distancias={"A":{},"B":{},"C":{}}
hoja_candidatos=wb["Distancias de Candidatos"]

candidato_actual=None

for fila in range(2,17):
    letra=hoja_candidatos.cell(fila,1).value
    bloque=hoja_candidatos.cell(fila,2).value
    sucursal=hoja_candidatos.cell(fila,3).value
    ida=hoja_candidatos.cell(fila,4).value
    vuelta=hoja_candidatos.cell(fila,5).value
    
    if letra:
        candidato_actual=str(letra).strip().upper()
        
    if candidato_actual not in Candidato_distancias:
        continue
    
    if bloque is None or sucursal is None:
        continue
    
    ida=numero(ida)
    vuelta=numero(vuelta)
    
    if ida is None or vuelta is None:
        continue
    
    bloque=int(bloque)
    sucursal=str(sucursal).strip()
    Candidato_distancias[candidato_actual][bloque]={"sucursal":sucursal,"ida":ida,"vuelta":vuelta,"promedio":(ida+vuelta)/2}
    
#Aqui se agregan los candidatos como vertices y conectarlos con la entrada de cada bloque
for letra,datos_bloques in Candidato_distancias.items():
    candidato=nombre_candidato(letra)
    agregar_vertice(candidato)
    
    for numero_bloque,datos in datos_bloques.items():
        entrada=datos["sucursal"]
        agregar_conexion(candidato,entrada,datos["ida"],datos["vuelta"])
        
        

## 6. Algoritmo de Dijkstra

In [6]:
#5 dijkstra
#Dijkstra usando como peso:  peso = (ida + vuelta) / 2  
#Devuelve: distancias -> distancia minima promedio   anteriores -> permite reconstruir el camino
def dijkstra(grafo_local,inicio):

    distancias={vertice:inf for vertice in grafo_local}
    anteriores={vertice:None for vertice in grafo_local}
    
    distancias[inicio]=0.0
    cola=[(0.0,inicio)]
    
    while cola:
        distancia_actual,actual=heapq.heappop(cola)
        
        if distancia_actual>distancias[actual]:
            continue
        
        for vecino,datos in grafo_local[actual].items():
            peso=datos["peso"]
            nueva_distancia=distancia_actual+peso
            
            if nueva_distancia<distancias[vecino]:
                distancias[vecino]=nueva_distancia
                anteriores[vecino]=actual
                heapq.heappush(cola,(nueva_distancia,vecino))
                
    return distancias,anteriores


#6 reconstruir la ruta 

def reconstruir_ruta(anteriores,inicio,destino):
    ruta=[]
    actual=destino
    
    while actual is not None:
        ruta.append(actual)
        if actual==inicio:
            break
        actual=anteriores.get(actual)
        
    if not ruta or ruta[-1]!=inicio:
        return[]
    
    ruta.reverse()
    return ruta

#calcular ida y vuelta sobre la ruta de dijkstra
#La ruta elegida por Dijkstra usa el promedio de cada conexion.
#Despues se suman por separado las distancias de ida y vuelta.

def calcular_ida_vuelta(ruta):
    total_ida=0.0
    total_vuelta=0.0
    
    for actual,siguiente in zip(ruta,ruta[1:]):
        datos=grafo[actual][siguiente]
        total_ida+=datos["ida"]
        total_vuelta+=datos["vuelta"]
        
    promedio=(total_ida+total_vuelta)/2
    
    return total_ida,total_vuelta,promedio



## 7. Análisis de Dijkstra por candidato y bloque

In [7]:
#Aqui se crea cada subgrafo de cada bloque
#Crea el grafo correspondiente a un bloque.
#Incluye:  - Las 12 farmacias del bloque. 
# - El candidato.
# - La conexiin del candidato con la entrada del bloque.

def crear_subgrafo_bloque(bloque,candidato):
    nodos=set(bloques[bloque])
    nodos.add(candidato)
    subgrafo={nodo:{}for nodo in nodos}
    
    for u in nodos:
        for v,datos in grafo.get(u,{}).items():
            if v in nodos:
                subgrafo[u][v]=datos
                
    return subgrafo


#Aqui se analiza candidato + bloque
def analizar_bloque(letra_candidato,numero_bloque):
    candidato=nombre_candidato(letra_candidato)
    bloque=f"Bloque {numero_bloque}"
    
    subgrafo=crear_subgrafo_bloque(bloque,candidato)
    distancias,anteriores=dijkstra(subgrafo,candidato)
    resultados=[]
    
    for farmacia in bloques[bloque]:
        distancia_dijkstra=distancias.get(farmacia,inf)
        
        if distancia_dijkstra==inf:
            resultados.append({
                "farmacia": farmacia,
                "dijkstra": inf,
                "ida": inf,"vuelta": inf,"promedio": inf,"ruta": [],})
            continue
        
        ruta=reconstruir_ruta(anteriores,candidato,farmacia)
        ida,vuelta,promedio=calcular_ida_vuelta(ruta)
        resultados.append({
                        "farmacia": farmacia,
                        "dijkstra": distancia_dijkstra,
                        "ida": ida,"vuelta": vuelta,"promedio": promedio,
                        "ruta": ruta,})
        
    resultados.sort(key=lambda x:x["promedio"])
    return resultados


## 8. Verificación de Dirac y Ore

In [8]:
#VERIFICACION de condiciciones hamiltonianas


#Obtiene unicamente las farmacias pertenecientes al bloque.  El candidato NO forma parte del circuito Hamiltoniano.

def obtener_subgrafo_bloque(bloque):
    nodos=set(bloques[bloque])
    subgrafo={nodo:{}for nodo in nodos}
    for u in nodos:
        for v,datos in grafo.get(u,{}).items():
            if v in nodos:
                subgrafo[u][v]=datos
                
    return subgrafo

#Calcula el grado de cada vertice del subgrafo.
def calcular_grados(subgrafo):
    grados={}
    for vertice in subgrafo:
        grados[vertice]=len(subgrafo[vertice])
        
    return grados

#Verifica la condicion suficiente de Dirac. Un grafo simple con n >= 3 vertices cumple Dirac si:grado minimo >= n / 2
#si se cumple, el grafo es Hamiltoniano.
def verificar_dirac(subgrafo):
    n=len(subgrafo)
    grados=calcular_grados(subgrafo)
    
    if n<3:
        return{"cumple":False,"n":n,"grado_minimo":min(grados.values())if grados else 0}
    
    grado_minimo=min(grados.values())
    cumple=grado_minimo>=n/2
    
    return{"cumple":cumple,"n":n,"grado_minimo":grado_minimo}

#Verifica la condicion suficiente de Ore.  Para todo par de vertices NO adyacentes:  grado(u) + grado(v) >= n
def verificar_ore(subgrafo):
    
    n=len(subgrafo)
    grados=calcular_grados(subgrafo)
    pares_evaluados=[]
    cumple=True
    vertices=list(subgrafo.keys())
    for i in range(len(vertices)):
        for j in range(i+1,len(vertices)):
            u=vertices[i]
            v=vertices[j]
            
            #solo interesa revisar pares no conectados
            if v not in subgrafo[u]:
                suma_grados=grados[u]+grados[v]
                
                pares_evaluados.append((u,v,grados[u],grados[v],suma_grados))
                
                if suma_grados<n:
                    cumple=False

    return{"cumple":cumple,"n":n,"pares":pares_evaluados}


#10 mostrar resultados
def imprimir_linea():# XD
    print("="*90)
    
def imprimir_verificacion_hamiltoniana(numero_bloque):
    
    bloque=f"Bloque {numero_bloque}"
    subgrafo=obtener_subgrafo_bloque(bloque)
    
    grados=calcular_grados(subgrafo)
    resultado_dirac=verificar_dirac(subgrafo)
    resultado_ore=verificar_ore(subgrafo)
    
    print("\n"+"="*90)
    print(f"VERIFICACIÓN HAMILTONIANA - {bloque.upper()}")
    print("=" * 90)
    
    print("\nFarmacias del bloque:")
    
    for vertice in sorted(grados):
        print(f"  {vertice}: grado = {grados[vertice]}")
    
    print("\n" + "-" * 90)
    print("TEOREMA DE DIRAC")
    print("-" * 90)
    
    print(f"Numero de vertices: {resultado_dirac['n']}")
    print(f"Grado minimo:       {resultado_dirac['grado_minimo']}")
    print(f"n / 2:              {resultado_dirac['n'] / 2:.1f}")
    
    if resultado_dirac["cumple"]:
        print("Resultado: CUMPLE LA CONDICION DE DIRAC")
        print("Conclusion: el bloque tiene garantizada la existencia de")
        print("un circuito Hamiltoniano mediante el Teorema de Dirac.")
    else:
        print("Resultado: NO CUMPLE LA CONDICION DE DIRAC")
        print("Esto NO significa que no exista un circuito Hamiltoniano.")
    
    print("\n" + "-" * 90)
    print("TEOREMA DE ORE")
    print("-" * 90)
    
    if resultado_ore["cumple"]:
        print("Resultado: CUMPLE LA CONDICION DE ORE")
        print("Conclusión: el bloque tiene garantizada la existencia de")
        print("un circuito Hamiltoniano mediante el Teorema de Ore.")
    else:
        print("Resultado: NO CUMPLE LA CONDICION DE ORE")
        print("Esto NO significa que no exista un circuito Hamiltoniano.")
        
        

## 9. Backtracking para el circuito Hamiltoniano

In [9]:
# 11 busqueda 
# Busca un circuito Hamiltoniano utilizando backtracking.

#    Un circuito Hamiltoniano debe:
#    1. Visitar todos los vértices exactamente una vez.
#    2. No repetir vertices durante el recorrido.
#    3. Regresar al vértice inicial.
    
def buscar_circuito_hamiltoniano(subgrafo):
    vertices=list(subgrafo.keys())
    if len(vertices)<3:
        return[]
    
    #se elige una farmacia como punto inicial
    inicio=vertices[0]
    ruta=[inicio]
    visitados={inicio}
    
    
    def backtracking(actual):
        
        # Si ya visitamos todos los vértices,comprobamos si podemos regresar al inicio.
        if len(ruta)==len(vertices):
            if inicio in subgrafo[actual]:
                return True
            
            return False
        
        #se revisa los vecinos del vertice actual
        for vecino in subgrafo[actual]:
            
            #si la farmacia todavia no ha sido visitada
            if vecino not in visitados:
                
                #agregar a la ruta
                ruta.append(vecino)
                visitados.add(vecino)
                
                #continuar buscando
                if backtracking(vecino):
                    return True
                
                #si esta opcion no funciona, se deshace la decision
                ruta.pop()
                visitados.remove(vecino)
            
        return False
    
    #iniciar busqueda
    encontrado=backtracking(inicio)
    
    if encontrado:
        #se cierra el circuito regresando al inicio
        ruta.append(inicio)
        return ruta
    
    return[]


## 10. Verificación y distancia del circuito Hamiltoniano

In [10]:
#Verifica que un circuito encontrado sea Hamiltoniano.
def verificar_circuito_hamiltoniano(circuito,subgrafo):
    
    if not circuito:
        return False
    
    #El primer y ultimo vértice deben ser iguales
    if circuito[0]!=circuito[-1]:
        return False
    
    #Quito el ultimo elemento para contar solamente los vertices visitados.
    vertices_visitados=circuito[:-1]
    
    #Todos los vertices deben ser diferentes
    if len(vertices_visitados) != len(set(vertices_visitados)):
        return False
    
    #se debe visitar todos los vertices del bloque
    if set(vertices_visitados)!=set(subgrafo.keys()):
        return False
    
    #se verifica que cada conexion exista
    for actual,siguiente in zip(circuito,circuito[1:]):
        if siguiente not in subgrafo[actual]:
            return False
        
    return True


#  Calcula la distancia total de ida, vuelta y promedio del circuito Hamiltoniano.
def calcular_distancia_circuito(circuito):
    
    total_ida=0.0
    total_vuelta=0.0
    
    for actual,siguiente in zip(circuito,circuito[1:]):
        datos=grafo[actual][siguiente]
        total_ida+=datos["ida"]
        total_vuelta+=datos["vuelta"]
        
    promedio=(total_ida+total_vuelta)/2
    
    return total_ida,total_vuelta,promedio


def analizar_circuitos_hamiltonianos():

    resultados={}

    print("\n")
    imprimir_linea()
    print("CIRCUITOS HAMILTONIANOS DE LOS 5 BLOQUES")
    imprimir_linea()

    for numero_bloque in range(1, 6):

        bloque=f"Bloque {numero_bloque}"
        subgrafo=obtener_subgrafo_bloque(bloque)
        circuito=buscar_circuito_hamiltoniano(subgrafo)

        print("\n" + "=" * 90)
        print(f"{bloque.upper()}")
        print("=" * 90)

        if not circuito:
            print("\nNo se encontro un circuito Hamiltoniano.")
            resultados[numero_bloque]={"circuito": [],"valido": False}
            continue

        valido=verificar_circuito_hamiltoniano(circuito,subgrafo)

        print("\nCircuito encontrado:")

        print(" -> ".join(circuito))

        print("\nCantidad de vértices visitados:")

        print(len(circuito)-1)

        print("\n¿Es un circuito Hamiltoniano?")

        if valido:
            print("SÍ")
        else:
            print("NO")

        if valido:

            ida,vuelta,promedio=calcular_distancia_circuito(circuito)

            print("\nDistancia del circuito:")
            print(f"  Ida:      {ida:.2f} km")
            print(f"  Vuelta:   {vuelta:.2f} km")
            print(f"  Promedio: {promedio:.2f} km")

            resultados[numero_bloque]={"circuito": circuito,"valido": True,"ida": ida,"vuelta": vuelta,"promedio": promedio}

    return resultados

    

## 11. Funciones para mostrar los resultados de Dijkstra

In [11]:
def imprimir_resultado_bloque(letra_candidato, numero_bloque, resultados):
    info=datos_candidato(letra_candidato)
    candidato=info["nombre"]
    bloque=f"Bloque {numero_bloque}"

    imprimir_linea()
    print(f"{candidato.upper()} - {info['zona'].upper()}")
    print(f"Ubicacion: {info['ubicacion']}")
    print(f"Análisis: {bloque.upper()}")
    imprimir_linea()

    print("\nLAS 12 FARMACIAS ORDENADAS DE MENOR A MAYOR DISTANCIA\n")

    for posicion, resultado in enumerate(resultados, start=1):
        print(f"{posicion:2}. {resultado['farmacia']}")
        print(f"    Dijkstra (promedio): {formato_km(resultado['dijkstra'])}")
        print(f"    Ida:                 {formato_km(resultado['ida'])}")
        print(f"    Vuelta:              {formato_km(resultado['vuelta'])}")
        print(f"    Promedio ida/vuelta: {formato_km(resultado['promedio'])}")

        if resultado["ruta"]:
            print("    Ruta:")
            print("      " + " -> ".join(resultado["ruta"]))
        else:
            print("    Ruta: NO DISPONIBLE")

        print()

    imprimir_linea()
    print("LAS 2 FARMACIAS MÁS CERCANAS")
    imprimir_linea()

    for posicion, resultado in enumerate(resultados[:2], start=1):
        print(f"{posicion}. {resultado['farmacia']}")
        print(f"   Promedio ida/vuelta: {formato_km(resultado['promedio'])}")
        print(f"   Ida:                 {formato_km(resultado['ida'])}")
        print(f"   Vuelta:              {formato_km(resultado['vuelta'])}")




## 12. Ejecutar la parte Hamiltoniana

Primero verificamos Dirac y Ore para los cinco bloques y después buscamos un circuito Hamiltoniano mediante backtracking.

> Si un bloque no cumple Dirac u Ore, eso **no demuestra que no exista** un circuito Hamiltoniano; por eso ejecutamos el algoritmo de búsqueda.


In [12]:
for numero_bloque in range(1, 6):
    imprimir_verificacion_hamiltoniana(numero_bloque)

resultados_hamiltonianos = analizar_circuitos_hamiltonianos()



VERIFICACIÓN HAMILTONIANA - BLOQUE 1

Farmacias del bloque:
  33 Calle Col Guillen: grado = 3
  SIMÁN PLAZA VICAY: grado = 3
  Simán Arenales: grado = 4
  Simán Calpules: grado = 3
  Simán Calpules 2: grado = 3
  Simán Ivihsa: grado = 3
  Simán Luisiana: grado = 3
  Simán Pradera: grado = 3
  Simán Santa Martha: grado = 3
  Simán Zip San Jose: grado = 1
  Town Center: grado = 2
  VILLA OLIMPICA: grado = 3

------------------------------------------------------------------------------------------
TEOREMA DE DIRAC
------------------------------------------------------------------------------------------
Numero de vertices: 12
Grado minimo:       1
n / 2:              6.0
Resultado: NO CUMPLE LA CONDICION DE DIRAC
Esto NO significa que no exista un circuito Hamiltoniano.

------------------------------------------------------------------------------------------
TEOREMA DE ORE
------------------------------------------------------------------------------------------
Resultado: NO CUMPLE L

## 13. Ejecutar Dijkstra para los 3 candidatos

Para cada candidato analizamos los 5 bloques. Dentro de cada bloque se muestran las 12 farmacias ordenadas por distancia y las 2 más cercanas.


In [13]:
# ------------------------------------------------------------
# 11. EJECUCION PRINCIPAL
# ------------------------------------------------------------
print("\n")
imprimir_linea()
print("       PROYECTO DIJKSTRA - FARMACIAS SIMAN SPS")
imprimir_linea()
print("\nDatos cargados directamente desde:")
print(f"{ARCHIVO_EXCEL.resolve()}\n")

print("ESTRUCTURA DEL PROYECTO")
print("-" * 90)
print("Cantidad de farmacias por bloque:")
for bloque, farmacias in bloques.items():
    print(f"  {bloque}: {len(farmacias)} farmacias")

print("\nCANDIDATOS")
print("-" * 90)
for letra in ["A", "B", "C"]:
    info = datos_candidato(letra)
    print(f"{info['nombre']} - {info['zona']}")
    print(f"  Ubicación: {info['ubicacion']}")

print("\nMetodo de cálculo:")
print("  • Dijkstra utiliza el promedio de ida/vuelta como peso de cada conexion.")
print("  • La ruta se obtiene unicamente mediante las conexiones del grafo.")
print("  • Luego se calculan por separado la distancia de ida, vuelta y promedio.")

# Guardaremos los resultados generales para el resumen.
resumen = {"A": {}, "B": {}, "C": {}}

for letra in ["A", "B", "C"]:
    for numero_bloque in range(1, 6):
        resultados = analizar_bloque(letra, numero_bloque)
        resumen[letra][numero_bloque] = resultados
        imprimir_resultado_bloque(letra, numero_bloque, resultados)





       PROYECTO DIJKSTRA - FARMACIAS SIMAN SPS

Datos cargados directamente desde:
C:\Users\royum\Downloads\Proyecto_Discretas\Datos_SucursalesSimán.xlsx

ESTRUCTURA DEL PROYECTO
------------------------------------------------------------------------------------------
Cantidad de farmacias por bloque:
  Bloque 1: 12 farmacias
  Bloque 2: 12 farmacias
  Bloque 3: 12 farmacias
  Bloque 4: 12 farmacias
  Bloque 5: 12 farmacias

CANDIDATOS
------------------------------------------------------------------------------------------
Candidato A - Zona Norte
  Ubicación: Sector La 105 Brigada
Candidato B - Zona Sur
  Ubicación: Zona industrial, Sector el Polvorín
Candidato C - Zona Centro
  Ubicación: Barrio Los Andes, 21102 San Pedro Sula, Cortés, Honduras

Metodo de cálculo:
  • Dijkstra utiliza el promedio de ida/vuelta como peso de cada conexion.
  • La ruta se obtiene unicamente mediante las conexiones del grafo.
  • Luego se calculan por separado la distancia de ida, vuelta y promedio.

## 14. Resumen general

In [14]:
# ------------------------------------------------------------
# 12. RESUMEN GENERAL
# ------------------------------------------------------------
imprimir_linea()
print("RESUMEN GENERAL")
imprimir_linea()

for letra in ["A", "B", "C"]:
    info = datos_candidato(letra)
    print(f"\n{info['nombre'].upper()} - {info['zona'].upper()}")
    print(f"Ubicacion: {info['ubicacion']}")
    print("-" * 90)

    for numero_bloque in range(1, 6):
        resultados = resumen[letra][numero_bloque]
        validos = [r for r in resultados if r["promedio"] != inf]

        if not validos:
            print(f"Bloque {numero_bloque}: sin ruta disponible")
            continue

        mas_cercana = validos[0]

        print(
            f"Bloque {numero_bloque}: "
            f"{mas_cercana['farmacia']} | "
            f"promedio = {formato_km(mas_cercana['promedio'])} | "
            f"ida = {formato_km(mas_cercana['ida'])} | "
            f"vuelta = {formato_km(mas_cercana['vuelta'])}"
        )

print("\n")
imprimir_linea()
print("FIN DEL ANÁLISIS")
imprimir_linea()

RESUMEN GENERAL

CANDIDATO A - ZONA NORTE
Ubicacion: Sector La 105 Brigada
------------------------------------------------------------------------------------------
Bloque 1: 33 Calle Col Guillen | promedio = 8.90 km | ida = 8.80 km | vuelta = 9.00 km
Bloque 2: Rancho El Coco | promedio = 3.60 km | ida = 3.60 km | vuelta = 3.60 km
Bloque 3: Barandillas | promedio = 2.60 km | ida = 2.20 km | vuelta = 3.00 km
Bloque 4: El Roble- Plaza Bogran | promedio = 4.00 km | ida = 3.90 km | vuelta = 4.10 km
Bloque 5: Simán 3 Ave. - Centro SPS | promedio = 2.30 km | ida = 2.20 km | vuelta = 2.40 km

CANDIDATO B - ZONA SUR
Ubicacion: Zona industrial, Sector el Polvorín
------------------------------------------------------------------------------------------
Bloque 1: 33 Calle Col Guillen | promedio = 1.05 km | ida = 1.20 km | vuelta = 0.90 km
Bloque 2: Rancho El Coco | promedio = 10.90 km | ida = 10.80 km | vuelta = 11.00 km
Bloque 3: Barandillas | promedio = 6.85 km | ida = 7.70 km | vuelta = 6.00